# Task
Analyse the provided transaction dataset in Python and perform a structured data quality audit.

# What you should do
- Briefly describe the dataset and its possible business use case.
- Identify data quality issues and map them to relevant dimensions such as completeness, uniqueness, validity, consistency, and integrity.
- Compute at least 3 KPIs such as Completeness Rate, Error Rate, or Duplication Rate.
- Define at least 3 validation rules to detect problematic records automatically.
- Prepare a short audit summary with issue type, affected rows, severity, and recommended next action.
- Suggest suitable cleaning steps for the next chapter, but do not implement full cleaning yet.

# Submit
- Completed notebook
- KPI summary and short interpretation
- Validation rules and audit findings
- Recommended cleaning actions for next week

In [1]:
import pandas as pd

# Task 1: Dataset description
Briefly describe the dataset and its possible business use case.

In [2]:
df = pd.read_csv("../../data/week2_customer_transactions_messy.csv")
print('Shape:', df.shape)
df.head(10)

Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## Dataset Description
The dataset records customer payment transactions and contains 9 columns: transaction_id, customer_id, transaction_date, amount, currency, payment_method, status, region, and last_updated. The sample covers 10 rows across regions DE, FR, US, and NL, with amounts in EUR and USD.

## Business use case:
A payments or finance team could use this data to monitor transaction volumes, track payment method adoption, and audit transaction statuses across regions. The status column (completed, pending, cancelled) alongside amount and region could feed operational dashboards or fraud detection pipelines.

# Task 2: Issues by dimension
Identify data quality issues and map them to relevant dimensions such as completeness, uniqueness, validity, consistency, and integrity.

In [3]:
# Count missing values per column
completeness = df.isna().sum().rename('missing_count').to_frame()
completeness['affected_rows'] = [df.index[df[col].isna()].tolist() for col in df.columns]
completeness

,missing_count,affected_rows
transaction_id,0,[]
customer_id,1,[3]
transaction_date,0,[]
amount,1,[9]
currency,0,[]
payment_method,1,[10]
status,0,[]
region,0,[]
last_updated,1,[9]


In [4]:
# Count duplicate values per column on relevant columns
dup_mask = df.duplicated(subset=['transaction_id'], keep=False)

uniqueness = pd.DataFrame({
    'duplicate_count': {'transaction_id': int(dup_mask.sum())},
    'affected_rows': {'transaction_id': df.index[dup_mask].tolist()}
})
uniqueness

,duplicate_count,affected_rows
transaction_id,2,"[5, 6]"


In [5]:
# Each column checked against its own rules where applicable

valid_currencies = {'EUR', 'USD', 'GBP'}

# For transaction_date validity: format must be YYYY-MM-DD AND the date must actually exist
parsed_dates = pd.to_datetime(df['transaction_date'], errors='coerce', format='%Y-%m-%d')
correct_format = df['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
impossible_date = correct_format & parsed_dates.isna()

amount_numeric = pd.to_numeric(df['amount'], errors='coerce')

validity_masks = {
    'transaction_date': impossible_date,
    'amount': (amount_numeric < 0) | (amount_numeric == 0) | (amount_numeric > 100_000),
    'currency': ~df['currency'].str.upper().isin(valid_currencies),
}

validity = pd.DataFrame({
    'violation_count': {col: int(mask.sum()) for col, mask in validity_masks.items()},
    'affected_rows': {col: df.index[mask].tolist() for col, mask in validity_masks.items()}
})
validity

,violation_count,affected_rows
transaction_date,1,[7]
amount,3,"[1, 2, 8]"
currency,1,[4]


In [6]:
# Check for formatting inconsistencies (valid value, wrong form)
std_date_format = df['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)

consistency_masks = {
    'transaction_date': ~std_date_format,
    'currency': df['currency'] != df['currency'].str.upper(),
    'payment_method': df['payment_method'] != df['payment_method'].str.lower(),
    'status': df['status'] != df['status'].str.lower(),
    'region': df['region'] != df['region'].str.upper(),
}

consistency = pd.DataFrame({
    'violation_count': {col: int(mask.sum()) for col, mask in consistency_masks.items()},
    'affected_rows': {col: df.index[mask].tolist() for col, mask in consistency_masks.items()}
})
consistency

,violation_count,affected_rows
transaction_date,2,"[1, 2]"
currency,0,[]
payment_method,2,"[1, 10]"
status,0,[]
region,1,[1]


In [7]:
# Referential and temporal relationship checks
# last_updated before transaction_date is only checked where transaction_date parsed successfully

last_updated_parsed = pd.to_datetime(df['last_updated'], errors='coerce')
transaction_date_parsed = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
date_parsed_ok = transaction_date_parsed.notna()

integrity_masks = {
    'customer_id': df['customer_id'].isna(),
    'last_updated': (last_updated_parsed < transaction_date_parsed) & date_parsed_ok,
}

integrity = pd.DataFrame({
    'violation_count': {col: int(mask.sum()) for col, mask in integrity_masks.items()},
    'affected_rows': {col: df.index[mask].tolist() for col, mask in integrity_masks.items()}
})
integrity

,violation_count,affected_rows
customer_id,1,[3]
last_updated,1,[2]


In [8]:
# Internal update lag: how many days between transaction_date and last_updated
# Only checked where both dates parsed successfully
# Rows where lag exceeds 7 days are flagged

valid_dates = transaction_date_parsed.notna() & last_updated_parsed.notna()
lag = (last_updated_parsed - transaction_date_parsed).dt.days
timeliness_mask = valid_dates & (lag > 7)

timeliness = pd.DataFrame({
    'violation_count': {'last_updated': int(timeliness_mask.sum())},
    'affected_rows': {'last_updated': df.index[timeliness_mask].tolist()}
})

timeliness

,violation_count,affected_rows
last_updated,3,"[1, 5, 6]"


Each column in the dataset was assessed against six data quality dimensions: completeness, uniqueness, validity, consistency, integrity, and timeliness. Rather than checking only the most obviously problematic columns, every column was evaluated against each dimension where the check was meaningful, for example, uniqueness was only applied to `transaction_id` since categorical columns such as `currency` or `status` are expected to repeat.

Completeness was checked across all columns for missing values. Uniqueness was restricted to `transaction_id` as the primary identifier. Validity rules were applied where constraints could be defined — impossible calendar dates for `transaction_date`, ISO 4217 conformance for `currency`, and numeric bounds for `amount`. Consistency checks flagged deviations from a standard form, such as non-canonical date formats and mixed casing in `currency`, `payment_method`, `status`, and `region`. Integrity checks covered the referential dependency of `transaction_id` on `customer_id`, and the temporal constraint that `last_updated` must not precede `transaction_date`.

For timeliness, external timeliness (whether data arrived within an expected ingestion window) cannot be assessed from a static file. However, internal update lag, measured as the number of days between `transaction_date` and `last_updated`, serves as a proxy. Records where this lag exceeds 7 days are flagged.

# Task 3: KPIs
Compute at least 3 KPIs such as Completeness Rate, Error Rate, or Duplication Rate.

In [9]:
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
completeness_rate = 1 - (missing_cells / total_cells)

print(f"Completeness Rate: {completeness_rate:.2%}")

Completeness Rate: 95.96%


In [10]:
duplication_rate = dup_mask.sum() / len(df)

print(f"Duplication Rate: {duplication_rate:.2%}")

Duplication Rate: 18.18%


In [11]:
validity_error_rate = pd.DataFrame(validity_masks).any(axis=1).mean()

print(f"Validity Error Rate: {validity_error_rate:.2%}")

Validity Error Rate: 45.45%


In [12]:
consistency_error_rate = pd.DataFrame(consistency_masks).any(axis=1).mean()

print(f"Consistency Error Rate: {consistency_error_rate:.2%}")

Consistency Error Rate: 27.27%


In [13]:
integrity_rate = pd.DataFrame(integrity_masks).any(axis=1).mean()

print(f"Integrity Rate: {integrity_rate:.2%}")

Integrity Rate: 18.18%


In [14]:
timeliness_rate = (~timeliness_mask & valid_dates).sum() / valid_dates.sum()

print(f"Timeliness Rate (<=7 days): {timeliness_rate:.2%}")

Timeliness Rate (<=7 days): 66.67%


In [16]:
# Total number of flagged cells across all dimensions combined

field_errors = (
        df.isna().sum().sum() +
        pd.DataFrame({'duplicate': dup_mask}).sum().sum() +
        pd.DataFrame(validity_masks).sum().sum() +
        pd.DataFrame(consistency_masks).sum().sum() +
        pd.DataFrame(integrity_masks).sum().sum() +
        pd.DataFrame({'timeliness': timeliness_mask}).sum().sum()
)

total_cells = len(df) * len(df.columns)
field_error_rate = field_errors / total_cells

print(f"Field Errors: {field_errors}")
print(f"Field Error Rate: {field_error_rate:.2%}")

Field Errors: 21
Field Error Rate: 21.21%


In [17]:
# All KPIs normalised to higher is better before averaging

overall = (
                  completeness_rate +
                  (1 - duplication_rate) +
                  (1 - validity_error_rate) +
                  (1 - consistency_error_rate) +
                  (1 - integrity_rate) +
                  timeliness_rate
          ) / 6

print(f"Overall Data Quality Score: {overall:.2%}")

Overall Data Quality Score: 75.59%


Six dimension-level KPIs were computed directly from the violation masks defined in Task 2, alongside a field error rate and an overall quality score. All KPIs were normalised to higher-is-better before being averaged into the overall score, meaning rates where lower values indicate better quality (e.g. duplication, validity error) were inverted as `(1 - rate)` before averaging.

The completeness rate of 95.96% indicates that four cells are missing across the dataset. The duplication rate of 18.18% is disproportionately high for a dataset of this size, caused entirely by the fully duplicated T0006 record. The validity error rate of 45.45% is the most concerning dimension-level finding, with five records carrying at least one validity violation across `transaction_date`, `amount`, and `currency`. The consistency error rate of 27.27% reflects three records with casing or format deviations, and the integrity rate of 18.18% corresponds to two records with referential or temporal violations. The timeliness rate of 66.67% indicates that three out of nine records with valid dates were updated within the seven-day threshold.

The field error rate of 21.21% counts 21 individually flagged cells out of 99 total, giving a cell-level view of data quality that complements the record-level rates above. The overall quality score of 75.59% is an equally weighted composite of all six dimension KPIs and should be interpreted cautiously given the small sample size of 11 records.

# Task 4: Audit Summary
Prepare a short audit summary with issue type, affected rows, severity, and recommended next action.

| Issue                                | Dimension    | Affected Rows | Severity | Severity Justification                                                                                                                                      | Recommended Next Action                                                                                |
|--------------------------------------|--------------|---------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|
| Duplicate transaction_id             | Uniqueness   | 2             | High     | Double counts revenue in operational dashboards and fraud detection pipelines                                                                               | Trace root cause in key-generation process; establish deduplication policy                             |
| Missing customer_id                  | Completeness | 1             | High     | Core identifier for the dataset; without it the transaction cannot be attributed to a customer, breaking warranty, compliance, and customer-level reporting | Escalate to source system owner; block ingestion of records without customer_id                        |
| Missing amount                       | Completeness | 1             | High     | Amount is the primary financial measure of this dataset; a missing value makes the record useless for revenue monitoring                                    | Investigate source system for incomplete writes; enforce non-null constraint at ingestion              |
| Impossible date 2026-02-30           | Validity     | 1             | High     | Record cannot be processed or placed on any timeline                                                                                                        | Reject at ingestion; enforce calendar validation at entry point                                        |
| Negative amount                      | Validity     | 1             | High     | Indicates a processing error during the transaction itself; a payment should not complete with a negative value unless explicitly classified as a refund    | Investigate transaction processing logs; introduce mandatory refund classification flag                |
| Outlier amount 1,000,000             | Validity     | 1             | High     | Amount is the core financial measure of this dataset; an erroneous value of this magnitude would corrupt revenue reporting and trigger false fraud signals  | Escalate to fraud and finance team for manual review; introduce amount ceiling validation              |
| Missing customer_id breaks join      | Integrity    | 1             | High     | Beyond analysis, an unattributed transaction means no customer can be held accountable; critical for warranty, compliance, and fraud traceability           | Same root fix as completeness; additionally quarantine record from any downstream joins until resolved |
| last_updated before transaction_date | Integrity    | 1             | High     | Indicates a systemic clock or pipeline ordering issue upstream; if present in more data this undermines the reliability of the entire audit trail           | Investigate pipeline timestamps; audit upstream system clocks for synchronisation issues               |
| Missing payment_method               | Completeness | 1             | Medium   | Payment method adoption tracking is a stated use case; missing values reduce coverage of that reporting                                                     | Investigate source system; enforce non-null constraint for payment_method at entry point               |
| Zero amount                          | Validity     | 1             | Medium   | A completed transaction with zero value is likely a processing error                                                                                        | Flag for business review; completed status should require a positive amount                            |
| Non-standard currency EURO           | Validity     | 1             | Medium   | Will fail downstream ISO 4217 currency validation and break multi-currency aggregations                                                                     | Enforce currency whitelist at ingestion; map EURO to EUR                                               |
| Non-standard date format 06-01-2026  | Consistency  | 1             | Medium   | Day/month ambiguity means the date could be misinterpreted by automated systems                                                                             | Enforce ISO 8601 format at entry point                                                                 |
| Update lag exceeds 7 days            | Timeliness   | 2             | Medium   | Stale records reduce reliability of operational dashboards                                                                                                  | Investigate update process; set automated alert for lag exceeding threshold                            |
| Missing last_updated                 | Completeness | 1             | Low      | Audit trail incomplete but does not affect core transaction analysis                                                                                        | Flag for monitoring; enforce at ingestion in next pipeline version                                     |
| Non-standard date format 2026/01/06  | Consistency  | 1             | Low      | Parseable but inconsistent; low risk if handled in cleaning                                                                                                 | Standardise to YYYY-MM-DD at entry point                                                               |
| Mixed case CARD                      | Consistency  | 1             | Low      | Breaks grouping on payment_method but easily corrected                                                                                                      | Normalise to lowercase at ingestion                                                                    |
| Mixed case region de                 | Consistency  | 1             | Low      | Breaks grouping on region but easily corrected                                                                                                              | Normalise to uppercase at ingestion                                                                    |

# Task 5: Suggestions
Suggest suitable cleaning steps for the next chapter, but do not implement full cleaning yet.

The cleaning steps outlined above address all issues that can be resolved programmatically without external input. Duplicate records can be dropped on `transaction_id`, and date parsing issues can be resolved by standardising to ISO 8601 using `pd.to_datetime` with `errors='coerce'`, which safely handles both non-standard formats and impossible dates by coercing failures to NaN. Categorical inconsistencies in `currency`, `payment_method`, and `region` are straightforward normalisation operations using `.str.upper()`, `.str.lower()`, and a targeted `.replace()` for the `EURO` case. The missing `payment_method` value can be filled with a placeholder to preserve the record without fabricating data.

The update lag issue does not require cleaning in the traditional sense — rather, a derived column capturing the lag in days should be retained for downstream monitoring and timeliness reporting.

Several issues identified in Task 4 fall outside the scope of programmatic cleaning. The negative amount, missing `customer_id`, missing `amount`, missing `last_updated`, the outlier of 1,000,000, and the `last_updated` before `transaction_date` case all require investigation at the source system or a business-level decision before any code should act on them. Applying a fix without that context risks introducing incorrect values that are harder to detect than the original problem. These records should be quarantined and escalated before the cleaning chapter proceeds.